In [1]:
import torch

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
m = AutoModelForCausalLM.from_pretrained(
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
    device_map="auto",
    torch_dtype="auto",
)
dm = AutoModelForCausalLM.from_pretrained(
    "DeepSeek-R1-Distill-Qwen-7B-W8A8-Dynamic-Per-Token",
    device_map="auto",
    torch_dtype="auto",
)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
Loading checkpoint shards: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:02<00:00,  1.48s/it]
/usr/local/lib/python3.10/dist-packages/pydantic/_internal/_fields.py:186: UserWarning: Field name "registry_requires_subclass" shadows an attribute in parent "RegistryMixin"; 
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/pydantic/_internal/_fields.py:186: UserWarning: Field name "registry_requires_subclass" shadows an attribute in parent "SparsityCompressionConfig"; 
  warnings.warn(
Loading checkpoint shards: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.07it/s]


In [5]:
pm = {k:v for k, v in m.named_parameters()}
pdm = {k:v for k, v in dm.named_parameters()}

In [9]:
pm.keys()

dict_keys(['model.embed_tokens.weight', 'model.layers.0.self_attn.q_proj.weight', 'model.layers.0.self_attn.q_proj.bias', 'model.layers.0.self_attn.k_proj.weight', 'model.layers.0.self_attn.k_proj.bias', 'model.layers.0.self_attn.v_proj.weight', 'model.layers.0.self_attn.v_proj.bias', 'model.layers.0.self_attn.o_proj.weight', 'model.layers.0.mlp.gate_proj.weight', 'model.layers.0.mlp.up_proj.weight', 'model.layers.0.mlp.down_proj.weight', 'model.layers.0.input_layernorm.weight', 'model.layers.0.post_attention_layernorm.weight', 'model.layers.1.self_attn.q_proj.weight', 'model.layers.1.self_attn.q_proj.bias', 'model.layers.1.self_attn.k_proj.weight', 'model.layers.1.self_attn.k_proj.bias', 'model.layers.1.self_attn.v_proj.weight', 'model.layers.1.self_attn.v_proj.bias', 'model.layers.1.self_attn.o_proj.weight', 'model.layers.1.mlp.gate_proj.weight', 'model.layers.1.mlp.up_proj.weight', 'model.layers.1.mlp.down_proj.weight', 'model.layers.1.input_layernorm.weight', 'model.layers.1.post_a

In [10]:
pdm.keys()

dict_keys(['model.embed_tokens.weight', 'model.layers.0.self_attn.q_proj.bias', 'model.layers.0.self_attn.q_proj.weight_scale', 'model.layers.0.self_attn.q_proj.weight', 'model.layers.0.self_attn.k_proj.bias', 'model.layers.0.self_attn.k_proj.weight_scale', 'model.layers.0.self_attn.k_proj.weight', 'model.layers.0.self_attn.v_proj.bias', 'model.layers.0.self_attn.v_proj.weight_scale', 'model.layers.0.self_attn.v_proj.weight', 'model.layers.0.self_attn.o_proj.weight_scale', 'model.layers.0.self_attn.o_proj.weight', 'model.layers.0.mlp.gate_proj.weight_scale', 'model.layers.0.mlp.gate_proj.weight', 'model.layers.0.mlp.up_proj.weight_scale', 'model.layers.0.mlp.up_proj.weight', 'model.layers.0.mlp.down_proj.weight_scale', 'model.layers.0.mlp.down_proj.weight', 'model.layers.0.input_layernorm.weight', 'model.layers.0.post_attention_layernorm.weight', 'model.layers.1.self_attn.q_proj.bias', 'model.layers.1.self_attn.q_proj.weight_scale', 'model.layers.1.self_attn.q_proj.weight', 'model.laye

In [16]:
pm['model.layers.0.self_attn.q_proj.weight']

Parameter containing:
tensor([[-9.7752e-06, -1.9165e-02,  1.2024e-02,  ..., -4.2725e-02,
         -7.6294e-03,  2.2125e-03],
        [ 1.2085e-02, -5.9326e-02,  9.8877e-03,  ..., -5.2246e-02,
          5.6763e-03,  2.7924e-03],
        [-1.0803e-02,  1.0681e-02,  3.9673e-03,  ..., -2.6733e-02,
         -1.0803e-02,  1.2939e-02],
        ...,
        [ 9.4604e-03,  9.8877e-03,  4.3945e-02,  ..., -3.2959e-02,
         -1.0498e-02, -5.0354e-03],
        [ 8.2397e-03, -2.5146e-02, -8.5449e-04,  ..., -3.3936e-02,
         -2.8809e-02,  1.2817e-02],
        [ 3.5645e-02, -2.1362e-02, -8.0490e-04,  ..., -3.1494e-02,
          1.6357e-02, -2.1362e-02]], device='cuda:0', dtype=torch.bfloat16,
       requires_grad=True)

In [15]:
pdm['model.layers.0.self_attn.q_proj.weight']

Parameter containing:
tensor([[  0, -10,   7,  ..., -21,  -4,   1],
        [  3, -14,   2,  ..., -12,   1,   1],
        [ -3,   3,   1,  ...,  -6,  -3,   3],
        ...,
        [  6,   7,  32,  ..., -21,  -7,  -2],
        [  8, -24,  -1,  ..., -30, -28,  11],
        [ 26, -16,  -1,  ..., -22,  12, -14]], device='cuda:0',
       dtype=torch.int8)

In [17]:
pdm['model.layers.0.self_attn.q_proj.weight'] / pm['model.layers.0.self_attn.q_proj.weight']

tensor([[  -0.,  520.,  584.,  ...,  492.,  524.,  452.],
        [ 248.,  236.,  202.,  ...,  230.,  176.,  358.],
        [ 278.,  280.,  252.,  ...,  224.,  278.,  232.],
        ...,
        [ 636.,  708.,  728.,  ...,  636.,  668.,  398.],
        [ 972.,  956., 1168.,  ...,  884.,  972.,  860.],
        [ 728.,  748., 1240.,  ...,  700.,  732.,  656.]], device='cuda:0',
       dtype=torch.bfloat16, grad_fn=<DivBackward0>)

### least square with zero alpha

In [42]:
beta = (pdm['model.layers.0.self_attn.q_proj.weight'] * pm['model.layers.0.self_attn.q_proj.weight']).sum(dim=-1, keepdim=True).float() / (pm['model.layers.0.self_attn.q_proj.weight'] ** 2).sum(dim=-1, keepdim=True)

In [43]:
(pm['model.layers.0.self_attn.q_proj.weight'] * beta).int()

tensor([[  0, -10,   6,  ..., -22,  -4,   1],
        [  2, -14,   2,  ..., -12,   1,   0],
        [ -2,   2,   0,  ...,  -6,  -2,   3],
        ...,
        [  6,   6,  28,  ..., -21,  -6,  -3],
        [  7, -22,   0,  ..., -30, -26,  11],
        [ 25, -15,   0,  ..., -22,  11, -15]], device='cuda:0',
       dtype=torch.int32)

In [44]:
beta

tensor([[524.7205],
        [243.2000],
        [249.8313],
        ...,
        [651.3778],
        [908.3871],
        [715.6685]], device='cuda:0', grad_fn=<DivBackward0>)

## using default scale

In [37]:
(pm['model.layers.0.self_attn.q_proj.weight'] / pdm['model.layers.0.self_attn.q_proj.weight_scale'] * 2).int()

tensor([[  0, -11,   7,  ..., -25,  -4,   1],
        [  3, -15,   2,  ..., -13,   1,   0],
        [ -3,   2,   1,  ...,  -7,  -3,   3],
        ...,
        [  7,   7,  33,  ..., -25,  -8,  -3],
        [  8, -26,   0,  ..., -36, -30,  13],
        [ 29, -17,   0,  ..., -26,  13, -17]], device='cuda:0',
       dtype=torch.int32)

## least square with alpha & beta

In [40]:
pdmw = pdm['model.layers.0.self_attn.q_proj.weight'].float()
pmw = pm['model.layers.0.self_attn.q_proj.weight']

pdmw = pdmw - pdmw.mean(dim=-1, keepdim=True)
pmw = pmw - pmw.mean(dim=-1, keepdim=True)

beta = (pdmw * pmw).sum(dim=-1, keepdim=True) / (pmw **2).sum(dim=-1, keepdim=True)
alpha = (pdm['model.layers.0.self_attn.q_proj.weight'] - pm['model.layers.0.self_attn.q_proj.weight'] * beta).mean(dim=-1, keepdim=True)

In [41]:
beta

tensor([[524.4738],
        [243.2699],
        [249.9417],
        ...,
        [652.8514],
        [908.6208],
        [715.0206]], device='cuda:0', grad_fn=<DivBackward0>)

In [45]:
alpha

tensor([[-0.0275],
        [-0.0240],
        [-0.0186],
        ...,
        [ 0.0100],
        [-0.0161],
        [-0.0866]], device='cuda:0', grad_fn=<MeanBackward1>)

In [47]:
(pm['model.layers.0.self_attn.q_proj.weight'] * beta + alpha).int() - pdm['model.layers.0.self_attn.q_proj.weight']

tensor([[ 0,  0, -1,  ..., -1,  0,  0],
        [-1,  0,  0,  ...,  0,  0, -1],
        [ 1, -1, -1,  ...,  0,  1,  0],
        ...,
        [ 0, -1, -4,  ...,  0,  1, -1],
        [-1,  2,  1,  ...,  0,  2,  0],
        [-1,  1,  1,  ...,  0, -1, -1]], device='cuda:0', dtype=torch.int32)

In [49]:
(pm['model.layers.0.self_attn.q_proj.weight'] * beta + alpha).int()

tensor([[  0, -10,   6,  ..., -22,  -4,   1],
        [  2, -14,   2,  ..., -12,   1,   0],
        [ -2,   2,   0,  ...,  -6,  -2,   3],
        ...,
        [  6,   6,  28,  ..., -21,  -6,  -3],
        [  7, -22,   0,  ..., -30, -26,  11],
        [ 25, -15,   0,  ..., -22,  11, -15]], device='cuda:0',
       dtype=torch.int32)